In [1]:
from datasets import load_dataset

ds = load_dataset("uqa/UQA") # splits : train , validation
print (ds)

ex = ds["train"][0]
print(ex.keys()) # id , title , context , question, answers
print(ex["question"])
print(ex["answer"]) # { ’ text ’: [...] , ’ answer_start ’: [...]}
n_total = len(ds["train"])
n_ans = sum(len(a.strip()) > 0 for a in ds["train"]["answer"])
print(f"train rows: {n_total}, answerable: {n_ans}")

d:\Anaconda\envs\uqa_qg\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'is_impossible', 'answer', 'answer_start'],
        num_rows: 124745
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'is_impossible', 'answer', 'answer_start'],
        num_rows: 16824
    })
})
dict_keys(['id', 'title', 'context', 'question', 'is_impossible', 'answer', 'answer_start'])
بیونس نے کب مقبولیت حاصل کرنا شروع کی؟
1990 کی دہائی کے آخر میں
train rows: 124745, answerable: 83018


In [2]:
import csv

ANS_OPEN, ANS_CLOSE = "<ans>", "</ans>"

# Urdu full stop U+06D4, Urdu question mark U+061F, exclamation mark
SENT_DELIMS = "\u06D4\u061F!"

def split_sentences(text):
    """Yield (start, end, sentence) with offsets into text."""
    start = 0

    for i, ch in enumerate(text):
        if ch in SENT_DELIMS:
            yield start, i + 1, text[start:i + 1]
            start = i + 1

    if start < len(text):
        yield start, len(text), text[start:]


def make_pair(example, max_src=60, max_tgt=25):
    """Return (source, target) or None if the row is unusable."""

    answer = example["answer"]
    a_start = example["answer_start"]

    if not answer or len(answer.strip()) == 0:
        return None

    a_text = answer
    a_end = a_start + len(a_text)

    context = example["context"]

    for s, e, sent in split_sentences(context):

        if s <= a_start < e:
            rel = a_start - s  # offset inside the sentence

            if sent[rel:rel + len(a_text)] != a_text:
                return None  # offset mismatch -> skip

            src = (
                sent[:rel]
                + " "
                + ANS_OPEN
                + " "
                + a_text
                + " "
                + ANS_CLOSE
                + " "
                + sent[rel + len(a_text):]
            ).strip()

            src = " ".join(src.split())  # normalize whitespace
            tgt = " ".join(example["question"].split())

            if len(src.split()) > max_src or len(tgt.split()) > max_tgt:
                return None

            return src, tgt

    return None


def build_split(split, out_path):
    pairs = [p for p in map(make_pair, split) if p is not None]

    with open(out_path, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(
            f,
            delimiter="\t",
            quoting=csv.QUOTE_NONE,
            escapechar="\\"
        )

        w.writerows(pairs)

    print(f"{out_path}: {len(pairs)} pairs")
    return pairs


train_pairs = build_split(ds["train"], "train.tsv")
valid_pairs = build_split(ds["validation"], "valid.tsv")

train.tsv: 75067 pairs
valid.tsv: 10018 pairs


In [3]:
import sentencepiece as spm

# One line per sentence: sources and targets from the TRAIN split only
with open("sp_corpus.txt", "w", encoding="utf-8") as f:
    for src, tgt in train_pairs:
        f.write(src + "\n" + tgt + "\n")



In [5]:
spm.SentencePieceTrainer.train(
    input="sp_corpus.txt",
    model_prefix="ur_sp",
    vocab_size=8000,
    model_type="unigram",
    character_coverage=1.0,  # keep every Urdu character
    user_defined_symbols=[ANS_OPEN, ANS_CLOSE],
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
)

sp = spm.SentencePieceProcessor(model_file="ur_sp.model")
PAD, UNK, BOS, EOS = 0, 1, 2, 3

src, tgt = train_pairs[0]

print(sp.encode(src, out_type=str))  # pieces
print(sp.encode(tgt))                 # ids
print(sp.decode(sp.encode(tgt)) == tgt)  # round-trip check

['▁ہیوسٹن', '▁،', '▁ٹیکساس', '▁میں', '▁پیدا', '▁ہوئی', '▁اور', '▁اس', '▁کی', '▁پرورش', '▁ہوئی', '▁،', '▁اس', '▁نے', '▁بچپن', '▁میں', '▁مختلف', '▁گانے', '▁اور', '▁رقص', '▁کے', '▁مقابلوں', '▁میں', '▁پرفارم', '▁کیا', '▁،', '▁اور', '▁', '<ans>', '▁1990', '▁کی', '▁دہائی', '▁کے', '▁آخر', '▁میں', '▁', '</ans>', '▁R', '&', 'B', '▁گر', 'ل', '▁گروپ', '▁ڈسٹنی', '▁چائلڈ', '▁کے', '▁لیڈ', '▁گلوکار', '▁کی', '▁حیثیت', '▁سے', '▁شہرت', '▁حاصل', '▁کی۔']
[2757, 18, 83, 2810, 100, 151, 99, 9, 11]
True


In [6]:
import sacrebleu
from rouge_score import rouge_scorer


def score(hyps, refs):
    bleu = sacrebleu.corpus_bleu(hyps, [refs]).score

    scorer = rouge_scorer.RougeScorer(
        ["rougeL"],
        use_stemmer=False
    )

    rl = sum(
        scorer.score(r, h)["rougeL"].fmeasure
        for h, r in zip(hyps, refs)
    ) / len(refs)

    unk_rate = sum(
        h.count("\u2047") for h in hyps
    ) / max(
        1,
        sum(len(h.split()) for h in hyps)
    )  # SP decodes <unk> as U+2047

    return {
        "BLEU-4": bleu,
        "ROUGE-L": rl,
        "unk_rate": unk_rate
    }